# GAE-GeometricAutoEncoder: CLI-Only Dual-GPU Runner

All model logic, 2-GPU sharding, checkpoint downloads, and BF16 half-precision are built into the codebase.

In [ ]:
# Cell 1: Clone / Sync Repository & Install Dependencies via uv pip
import os, subprocess

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
ROOT = "/kaggle/working/GAE-GeometricAutoEncoder"
BRANCH = "feat/kaggle-runner"
REPO_URL = "https://github.com/A511-git/GAE-GeometricAutoEncoder"

if not os.path.isdir(ROOT):
    print(f"--> Cloning {REPO_URL} (branch: {BRANCH})...")
    subprocess.run(f"git clone -b {BRANCH} {REPO_URL} {ROOT}", shell=True, check=True)
else:
    print(f"--> Syncing {ROOT} to origin/{BRANCH}...")
    subprocess.run(f"git -C {ROOT} fetch origin {BRANCH} && git -C {ROOT} reset --hard origin/{BRANCH}", shell=True, check=True)

print("--> Installing dependencies via uv pip...")
subprocess.run("pip install -q uv && uv pip install --system --reinstall numpy scipy", shell=True, check=True)
subprocess.run(f"uv pip install --system -r {ROOT}/requirements.txt", shell=True, check=True)

os.chdir(ROOT)
print("\n[SUCCESS] Environment ready!")

In [ ]:
# Cell 2: Run Zero-Manual-Pose Multi-View Re-framer via CLI
# Automatic weight downloading, 2-GPU parallel splitting, and BF16 precision are self-contained.

!PYTHONPATH=/kaggle/working/GAE-GeometricAutoEncoder python scripts/demo/multiview_reframer.py \
    --images /kaggle/input/datasets/newmailserver/test-img \
    --anchor PXL_20260926_175340689.jpg \
    --pull-back 1.2 \
    --fov-scale 0.85 \
    --resolution 378 672 \
    --precision bf16 \
    --output /kaggle/working/my_results

In [ ]:
# Cell 3: View Generated Novel View & List 3D Output Files
import cv2, matplotlib.pyplot as plt
from pathlib import Path

OUT = Path("/kaggle/working/my_results/scannetpp")
novel_path = OUT / "000_novel_view.png"

if novel_path.exists():
    img = cv2.cvtColor(cv2.imread(str(novel_path)), cv2.COLOR_BGR2RGB)
    plt.figure(figsize=(10, 6))
    plt.imshow(img)
    plt.title("Generated Novel Viewpoint (Pull-Back 1.2m, FOV 0.85)")
    plt.axis("off")
    plt.show()

print("\n[Generated Output Artifacts in:", str(OUT), "]")
if OUT.exists():
    for f in sorted(OUT.iterdir()):
        size_mb = f.stat().st_size / (1024 * 1024)
        print(f"  • {f.name} ({size_mb:.2f} MB)")